**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# MIMO Communications

Multiple antennas at both ends turn multipath — [Digital Comms'](./Digital_Communications.ipynb) villain — into **extra spectrum out of thin air**: parallel spatial channels through the same band. Capacity with many antennas, diversity vs multiplexing, and SVD precoding that turns the channel into independent pipes (verified: the pipes really are independent).

## 1. Pre-requisites

[Digital Communications](./Digital_Communications.ipynb), [Linear Algebra](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) S4 (SVD — the star of Session 3), [Information Theory](../Intro_Math/Information_Theory/Information_Theory.ipynb) S4.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

def qpsk_syms(n): return (rng.choice([-1,1],n) + 1j*rng.choice([-1,1],n))/np.sqrt(2)
def rayleigh(nr, nt): return (rng.standard_normal((nr,nt)) + 1j*rng.standard_normal((nr,nt)))/np.sqrt(2)

---
### 🕐 Session 1 of 3 — *MIMO Capacity* (~35 min)
**Goal:** why capacity grows LINEARLY with min(antennas): the log-det formula, simulated.
**Builds on:** [Information Theory](../Intro_Math/Information_Theory/Information_Theory.ipynb) S4. &nbsp; **Feeds into:** Session 2 (diversity).

---

## 2. Spectrum from Space

💡 **Intuition.** A rich-scattering channel matrix $H$ has $\min(n_t, n_r)$ meaningful singular directions — each an independent spatial pipe. Capacity $C = \log_2\det(I + \frac{\rho}{n_t} H H^H)$ therefore grows ~**linearly** in $\min(n_t, n_r)$ at high SNR, while a single antenna only ever gets $\log(1{+}\rho)$. Multipath, the enemy of the single-antenna link, is the *resource* here: no scattering ⇒ rank-1 $H$ ⇒ pipes collapse.

In [ ]:

# YOUR CODE HERE


**What just happened.** Ergodic capacity at 20 dB: **5.9 / 11.3 / 22.2 / 44.0** bits/s/Hz for 1, 2, 4, and 8 antennas. Roughly a doubling each time the antenna count doubles, against a SISO ceiling that sits flat no matter how many antennas the plot's x-axis claims.

**The asymmetry is the economic argument for MIMO.** Capacity is *logarithmic* in power — doubling transmit power at 20 dB buys you about one extra bit — and *linear* in antennas. So going from 1 to 8 antennas gains 38 bits/s/Hz, while achieving the same by raising power would require an increase of roughly $2^{38}$. That is why 5G and Wi-Fi kept adding antennas rather than transmitters, and it is the single most consequential fact in this workshop.

**Now the 7.4, which is more interesting than a clean 8 would be.** Perfect linear scaling predicts $C(8)/C(1) = 8$; we measured 7.4. The shortfall is real and the printed comment names it: total transmit power is divided across antennas, so each pipe sees $\rho/n_t$ instead of $\rho$. Per-antenna capacity drifts down accordingly — 5.90, 5.65, 5.55, 5.50 bits/s/Hz — because each pipe is individually a little weaker even as there are more of them.

Note the shape of that drift: it is *converging*, not collapsing. The loss per pipe is a logarithm of a shrinking factor while the pipe count grows linearly, so the linear term wins asymptotically. "Capacity scales linearly with antennas" is a statement about the slope at high SNR, with a constant that this simulation makes visible. Precision about which part of a scaling law is exact and which is asymptotic is worth more than the headline.

**Where the rank comes from — and the counterintuitive consequence.** Those pipes exist only because $H$ has $\min(n_t, n_r)$ meaningful singular values, and $H$ is full rank only because the environment scatters richly. A clean line-of-sight channel gives a rank-1 $H$: all the singular values but one collapse, the pipes vanish, and eight antennas buy array gain but no multiplexing. **A MIMO link works better in a cluttered room than in an open field.** Multipath, the villain of [Digital Communications](./Digital_Communications.ipynb), is here the resource being harvested.

**One caveat on what was measured.** Averaging over 2000 channel draws gives *ergodic* capacity — the long-run average across fading states, which is the right quantity when a codeword spans many channel realisations. It says nothing about any individual bad draw. For a latency-limited link the governing quantity is outage capacity, determined by the unlucky realisations rather than the mean, and those bad draws are exactly what Session 2 is about.

---
### 🕐 Session 2 of 3 — *Diversity: Never Fade Alone* (~40 min)
**Goal:** Rayleigh fading murders BER; independent branches resurrect it — the diversity-order slopes.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (SVD precoding).

---

## 3. Insurance Against Fades

💡 **Intuition.** On a fading channel the *average* SNR is fine; the *bad moments* kill you — BER is dominated by the probability the channel is in a deep fade, which falls only as $1/\rho$ (diversity order 1). With $L$ **independent** branches, all must fade together: $P \sim \rho^{-L}$ — the BER-vs-SNR slope steepens to $L$. Maximum-ratio combining ([matched filtering](./Statistical_Signal_Processing.ipynb) across antennas!) collects it optimally; Alamouti's space-time code famously buys transmit diversity with two antennas and zero channel knowledge.

In [ ]:

# YOUR CODE HERE


**What just happened.** Three BER curves, and the thing to read is not their height but their **slope**. On log-BER against dB-SNR, $L = 1$ tracks the dotted reference at roughly one decade of BER per 10 dB. $L = 2$ falls about twice as fast, and $L = 4$ about four times as fast. Diversity order *is* the slope.

That distinction matters more than it first appears. Diversity does not shift the curve downward by a fixed amount — it **tilts** it. So the benefit is not a constant number of decibels; it grows without limit as SNR rises. At 5 dB the curves are close together; by 25 dB they are separated by orders of magnitude. Any technique that changes a slope beats any technique that changes an offset, given enough SNR.

**Why fading is so punishing in the first place.** On an AWGN channel BER falls *exponentially* with SNR. On a Rayleigh channel it falls only as $1/\rho$ — one decade per 10 dB — because the error rate is dominated not by typical conditions but by the probability of a deep fade. The average SNR can be perfectly healthy while the link is unusable, and pouring in transmit power is a poor remedy: 10 dB more power buys a single decade.

Diversity attacks the actual cause. With $L$ independent branches, a deep fade requires *all* of them to fade at once, and independent rare events multiply: $P \sim \rho^{-L}$. You are not making any branch better — you are making simultaneous failure rarer.

**The word "independent" is load-bearing.** Diversity order $L$ requires the branches to fade independently. Antennas spaced much closer than half a wavelength see strongly correlated channels and deliver far less than the nominal order; two perfectly correlated branches give array gain — a downward shift — but *no* change in slope, because they always fade together. This is the entire reason antenna-spacing rules exist in real designs, and it is why the theoretical curves above are an upper bound on what hardware achieves.

**One line does the combining, and it is a familiar one.** `(h.conj() * y).sum(0)` is maximum-ratio combining: weight each branch by its own conjugate channel, so strong branches count more and weak ones are attenuated rather than discarded. That is the [matched filter](./Statistical_Signal_Processing.ipynb) applied across antennas instead of across time, and it is provably the optimal linear combiner in white noise.

**And the practical caveat.** MRC needs several *receive* antennas and channel knowledge at the receiver — fine for a base station, awkward for a handset. Alamouti's space-time block code obtains full transmit diversity from two *transmit* antennas with no channel knowledge at the transmitter at all, which is why it appears in essentially every cellular standard: it moves the antenna cost to the tower.

---
### 🕐 Session 3 of 3 — *SVD Precoding: the Channel, Diagonalized* (~40 min)
**Goal:** with channel knowledge, transform MIMO into independent parallel pipes — verified.
**Builds on:** Session 2; [Linear Algebra](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) S4.

---

## 4. The SVD Cashes Its Biggest Check

💡 **Intuition.** Write $H = U\Sigma V^H$. Precode with $V$ at the transmitter and receive with $U^H$: the end-to-end channel becomes $U^H H V = \Sigma$ — **diagonal**. Four antennas, four *independent* scalar channels with gains $\sigma_i$, zero cross-talk, no equalization. Then pour power by [water-filling](../Intro_Math/Information_Theory/Information_Theory.ipynb): more into strong pipes, none into hopeless ones. This is exactly how LTE/5G/Wi-Fi beamforming works when the channel is known.

In [ ]:
# ORACLE 1: effective channel is diagonal with the singular values
# ORACLE 2: the pipes are independent — cross-talk between streams ≈ 0

# YOUR CODE HERE


**What just happened.** Two checks, testing genuinely different claims.

**The algebraic oracle.** $\|U^HHV - \mathrm{diag}(\sigma)\|_\infty = 1.3\times10^{-15}$ — machine precision. Precoding with $V$ and combining with $U^H$ turns the $4\times4$ channel into a diagonal matrix, exactly. Four antennas have become **four independent scalar channels** with gains $[2.912, 2.226, 1.431, 0.327]$: no cross-talk, no equalisation, no interference cancellation, just four separate one-dimensional links sharing a band. This is the [SVD](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) doing the most practically valuable thing it does anywhere in this curriculum.

**The statistical oracle.** Algebra is one thing; real QPSK streams through a noisy channel are another. Each stream's own-correlation is essentially 1 and the maximum cross-talk correlation is around 0.01 — the pipes really are independent when driven with data.

Read that 0.01 properly rather than calling it "small". With 20,000 symbols, the sampling noise floor for a correlation estimate is $1/\sqrt{20000} \approx 0.007$. The measured values — 0.0117, 0.0055, 0.0074, 0.0127 — all sit *at* that floor, so they are statistically indistinguishable from zero. The residual is finite-sample noise, not leakage.

**Stream 3 is the informative row.** Its own-correlation is 0.989 where the others are 1.000, because its gain is $\sigma = 0.33$ against 2.91 for the strongest — nearly 9× less amplitude, so about 19 dB worse SNR at the same noise level. Nothing has gone wrong. The pipes genuinely differ in quality, sometimes by an order of magnitude, and that spread is not a defect to be fixed but a *resource allocation problem* to be solved. The next cell solves it.

**What it costs.** Everything here assumed $H$ known at both ends. The receiver can estimate it from pilots, but the *transmitter* needs it too — and getting it there means feedback, which costs overhead that scales with antenna count and goes stale as the channel changes. That is why this works beautifully for a stationary laptop on Wi-Fi and poorly at vehicular speeds, and why Session 2's diversity schemes, which need far less knowledge, remain the robust fallback. The workshop's real summary is a trade: more channel knowledge buys more performance, and knowledge is not free.

In [ ]:
# water-filling on the four pipes — at LOW SNR, where the choice matters
# (at high SNR water-filling ≈ equal power; the gains appear when power is scarce)

# YOUR CODE HERE


**What just happened.** Four pipes with gains $[8.48, 4.95, 2.05, 0.11]$ — a spread of nearly 80× between best and worst — and water-filling allocated power $[0.48, 0.40, 0.11, 0.00]$. The weakest pipe receives **exactly nothing**. Rate rises from 3.44 to 4.24 bits/s/Hz, a 23% gain, purely from deciding where to put power that was already available.

**The zero is the interesting entry.** Most people predict the weak pipe gets "a little"; the optimum is none at all. Water-filling pours power into a landscape whose depth is $1/\text{gain}$, and the fourth pipe is so shallow that it sits *above* the waterline — spending any power there would buy less rate than the same power spent elsewhere. The reframe worth taking away: sometimes the optimal allocation to a channel is zero, and declining to use hardware you already own is the correct engineering decision rather than waste.

Notice this is the exact opposite of the intuition that drives diversity in Session 2. There, every branch was valuable because you were insuring against fades. Here, with channel knowledge in hand, you already *know* which pipe is bad — so there is nothing to insure against and no reason to feed it.

**Be careful how you quote the 23%.** The comment in the cell flags the condition and it matters: this is measured at **low SNR**, with `P_total = N0 = 1`. When power is scarce, concentrating it matters a great deal. At high SNR the water level rises well above every pipe's floor, water-filling converges toward equal power, and the advantage shrinks toward nothing. So 23% is not a general figure for MIMO precoding — it is the gain in the regime where power is the binding constraint. The honest statement is directional: *water-filling matters when power is tight, and stops mattering when it is plentiful*.

**Why the algorithm is a bisection.** `waterfill` searches for the water level $\mu$ such that total allocated power equals the budget. The allocation $p_i = \max(\mu - 1/g_i, 0)$ is monotone in $\mu$, so bisection converges reliably — a clean example of a constrained optimisation with an interpretable Lagrange multiplier, since $\mu$ *is* the water level and the KKT conditions are visible as the $\max(\cdot, 0)$.

**Bringing the three sessions together.** Scattering builds rank; rank builds pipes (Session 1). Independent branches steepen the BER slope when you lack channel knowledge (Session 2). With channel knowledge, the SVD diagonalises the link and water-filling optimally feeds the resulting pipes (Session 3). The through-line is that each step buys performance with a different currency — environment, antennas, or knowledge — and knowing which you can afford is the actual engineering.

## 5. Conclusion

Scattering builds rank, rank builds pipes; independent branches steepen the BER slope (measured); and the SVD — with channel knowledge — diagonalizes the link into verified-independent scalar channels fed by water-filling. The [SVD](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) never worked harder.

---
## Where next

- [Array Processing](./Array_Processing.ipynb) — the same antennas, pointed at *directions* instead of *rank*.
- [Channel Coding](./Channel_Coding.ipynb) — the codes riding inside each pipe.